# Blysyrebatteriet

## Fra ekvivalent krets til svovelsyretransport, ladningsaksept og gassing

### Pilotprosjekt for Matematikk 1

Et blysyrebatteri reagerer på flere tidsskalaer:

- et momentant ohmsk spenningsfall,
- rask og langsom elektrodepolarisasjon,
- lokal tømming og diffusjon av svovelsyre,
- endring i ladetilstand,
- blokkering og gjenoppretting av aktiv overflate,
- begrenset ladningsaksept ved mangel på blyioner,
- gassproduksjon og oksygenrekombinasjon ved overlading.

Prosjektet bygger en modellstige:

1. **Trevolums syrenettverk:** lineær algebra, nullrom og diffusjonsmoder
2. **Ett RC-ledd:** skalar ODE, puls og relaksasjon
3. **Ekvivalent krets:** raske og langsomme spenningsbidrag
4. **Hybridmodell:** SOC, syretransport og aktiv overflate
5. **Fordypning:** blyioner, ladningsaksept, gassing og vannforbruk

### Læringsmål

Etter prosjektet skal du kunne

- bygge en transportmatrise som $B^TDB$,
- tolke nullrom og egenmoder fysisk,
- løse en skalar RC-ODE analytisk og numerisk,
- skrive flere spenningsbidrag som et vektor-ODE-system,
- formulere materialbalanser for tre elektrolyttvolumer,
- koble strøm, SOC, syrekonsentrasjon og terminalspenning,
- forklare spenningsgjenoppretting etter en belastningspuls,
- modellere redusert aktiv overflate ved utlading,
- bruke en numerisk Jacobimatrise og tolke egenverdier,
- skille mellom hovedreaksjonstrøm, ladningsaksept og sidereaksjoner.

### Viktig avgrensning

Dette er en redusert undervisningsmodell. Parameterne er valgt for å gi realistiske størrelsesordener og tydelige tidsskalaer, men er ikke identifisert for et bestemt kommersielt batteri. Modellen skal ikke brukes til dimensjonering, sikkerhetsanalyse, ladealgoritmer eller levetidsprognoser.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

F = 96485.33212       # C/mol
R_gass = 8.314462618  # J/(mol K)
T_ref = 298.15        # K

# Referansebatteriet og fortegn

Vi modellerer én 2 V-celle med nominell kapasitet

$$Q_N=50\ \mathrm{Ah}.$$

Fortegnskonvensjon:

- $I>0$: utlading
- $I<0$: lading

Terminalspenningen skrives

$$
\boxed{
U=U_{eq}-R_0I-u_- -u_+ -u_d-\eta_{Pb}.}
$$

Ved utlading gir positive overspenninger lavere terminalspenning. Ved lading skifter de dynamiske bidragene fortegn gjennom strømmen.

In [ ]:
Q_N_Ah = 50.0
Q_N_C = 3600.0*Q_N_Ah
eta_I = 0.98

R0 = 1.8e-3       # ohm
R_neg = 0.8e-3    # ohm
R_pos = 2.4e-3    # ohm
R_diff = 3.5e-3   # ohm

tau_neg = 0.08    # s, rask negativ elektrode
tau_pos = 8.0     # s, positiv elektrode
tau_diff = 600.0  # s, langsom transport/polarisasjon

C_neg = tau_neg/R_neg
C_pos = tau_pos/R_pos
C_diff = tau_diff/R_diff

# Del A: Svovelsyre som et transportnettverk

## A.1 Tre elektrolyttvolumer

Vi bruker middelverdier for syrekonsentrasjonen i

1. negativ elektrode, $c_-$
2. separator, $c_s$
3. positiv elektrode, $c_+$

Diffusjonsstrømmene modelleres lineært:

$$J_{-s}=d_{-s}(c_--c_s),$$

$$J_{s+}=d_{s+}(c_s-c_+).$$

Forbindelsesmatrisen er

$$
B_c=
\begin{pmatrix}
1&-1&0\\
0&1&-1
\end{pmatrix}.
$$

In [ ]:
B_c = np.array([
    [1.0, -1.0, 0.0],
    [0.0, 1.0, -1.0]
])

d_minus_sep = 1.8e-4  # L/s, effektiv transportparameter
d_sep_plus = 1.2e-4   # L/s
D_c = np.diag([d_minus_sep, d_sep_plus])

## Oppgave A1: Bygg transportmatrisen

Beregn

$$
\boxed{K_c=B_c^TD_cB_c.}
$$

Kontroller at matrisen er symmetrisk og at

$$K_c\mathbf1=0.$$

Forklar hvorfor diffusjon kan omfordele syre, men ikke endre den totale syremengden.

In [ ]:
K_c = ...

print("K_c =
", K_c)
print("Symmetrisk:", ...)
print("K_c @ 1 =", ...)
print("Egenverdier:", ...)

## A.2 Volummatrisen

Effektive elektrolyttvolumer:

$$V_-=0.42\ \mathrm L,\quad V_s=0.28\ \mathrm L,\quad V_+=0.38\ \mathrm L.$$

Volummatrisen er

$$
M_c=\operatorname{diag}(V_-,V_s,V_+).
$$

Uten reaksjon:

$$
\boxed{M_c\dot c=-K_cc.}
$$

In [ ]:
V_syremodell = np.array([0.42, 0.28, 0.38])
M_c = np.diag(V_syremodell)
A_c = -np.linalg.solve(M_c, K_c)

## Oppgave A2: Ulike volum og egenmoder

Finn egenverdier og egenvektorer til $A_c$. Identifiser:

- felles syremode
- langsom forskjellsmode
- rask forskjellsmode

Beregn tidskonstantene $-1/\lambda$ for de negative egenverdiene.

In [ ]:
egenverdier_c, P_c = ...
print("Egenverdier:", egenverdier_c)
print("Egenvektorer:
", P_c)

# Beregn bare tidskonstanter for negative egenverdier.

## A.3 Symmetrisk lignende matrise

Når volumene er ulike, er $A_c=-M_c^{-1}K_c$ ikke symmetrisk. Definer

$$
y=M_c^{1/2}c.$$

Da blir den lignende systemmatrisen

$$
\boxed{
A_{sym}=-M_c^{-1/2}K_cM_c^{-1/2}.}
$$

Den er symmetrisk og har de samme egenverdiene som $A_c$.

In [ ]:
M_halv = np.diag(np.sqrt(V_syremodell))
M_mhalv = np.diag(1/np.sqrt(V_syremodell))
A_sym = ...

print("A_sym symmetrisk:", ...)
print("Egenverdier A_c:", ...)
print("Egenverdier A_sym:", ...)

# Del B: Ett RC-ledd som skalar ODE

Et polarisasjonsledd beskrives med

$$
\boxed{C\dot u=I-\frac uR.}
$$

Ekvivalent:

$$
\dot u=-\frac u{RC}+\frac IC.
$$

Ved konstant strøm er likevekten

$$u^*=RI,$$

og tidskonstanten er

$$\tau=RC.$$

## Oppgave B1: Analytisk løsning

Bruk det positive elektrodeleddet og en utladestrøm på $I=40$ A. Start med $u(0)=0$.

Vis at

$$
\boxed{u(t)=RI+(u_0-RI)e^{-t/\tau}.}
$$

In [ ]:
I_puls = 40.0


def u_pos_analytisk(t, u0=0.0):
    return ...

print("Likevektsspenning for RC-leddet:", R_pos*I_puls, "V")

## Oppgave B2: Euler, belastningspuls og hvile

Bruk strømprofilen

$$
I(t)=
\begin{cases}
40\ \mathrm A,&0\le t<100\ \mathrm s,\\
0,&100\le t\le700\ \mathrm s.
\end{cases}
$$

Sammenlign flere tidssteg og plott oppbygging og relaksasjon av $u_+$.

In [ ]:
def strøm_puls(t):
    return 40.0 if 0.0 <= t < 100.0 else 0.0


def rc_pos_ode(t, u):
    I = strøm_puls(t)
    return -u/tau_pos + (R_pos/tau_pos)*I


def euler_skalar(f, y0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0, n*dt, n+1)
    y = np.zeros(n+1)
    y[0] = y0
    for k in range(n):
        y[k+1] = ...
    return t, y

# Del C: Ekvivalent krets med flere tidsskalaer

Tilstanden er

$$
\boxed{u=(u_-,u_+,u_d)^T.}
$$

Systemet er

$$
\boxed{
\dot u=A_uu+b_uI(t),}
$$

med

$$
A_u=
\operatorname{diag}
\left(-\frac1{\tau_-},-\frac1{\tau_+},-\frac1{\tau_d}\right).
$$

In [ ]:
A_u = np.diag([-1/tau_neg, -1/tau_pos, -1/tau_diff])
b_u = np.array([R_neg/tau_neg, R_pos/tau_pos, R_diff/tau_diff])


def krets_ode(t, u):
    return A_u@u+b_u*strøm_puls(t)

## Oppgave C1: Tre spenningsmoder

Simuler pulsen og plott hvert spenningsbidrag separat. Sammenlign:

- momentant ohmsk fall $R_0I$
- rask negativ elektrode
- langsommere positiv elektrode
- langsom diffusjonspolarisasjon

In [ ]:
def euler_system(f, x0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0, n*dt, n+1)
    X = np.zeros((n+1, len(x0)))
    X[0] = x0
    for k in range(n):
        X[k+1] = ...
    return t, X

# Simuler krets_ode fra nulltilstand.

## C.2 Enkel likevektsspenning og SOC

Vi bruker først

$$
\boxed{
U_{eq}(z)=1.95+0.18z+0.05\ln\left(\frac{z+0.02}{1.02-z}
\right).}
$$

Dette er en pedagogisk, glatt funksjon for én celle.

SOC utvikles etter

$$
\boxed{\dot z=-\frac{\eta_I I_{MR}}{Q_N}.}
$$

In [ ]:
def U_eq_soc(z):
    z_clip = np.clip(z, 1e-4, 1-1e-4)
    return 1.95+0.18*z_clip+0.05*np.log((z_clip+0.02)/(1.02-z_clip))

## Oppgave C2: Terminalspenning under puls

Hold først SOC konstant på $z=0.80$ og beregn

$$U=U_{eq}(z)-R_0I-u_--u_+-u_d.$$

Forklar hvilke deler av spenningsgjenopprettingen som skjer raskt og langsomt.

# Del D: Hybridmodell med syretransport og aktiv overflate

## D.1 Tilstander

Hovedmodellen bruker

$$
\boxed{
X=(z,u_-,u_+,u_d,c_-,c_s,c_+,a)^T.}
$$

Her er $a\in(0,1]$ en samlet aktiv-overflatefaktor.

## D.2 Reaksjonskilde for syre

Ved utlading forbrukes syre i begge elektrodene. Vi bruker den pedagogiske kildevektoren

$$
\boxed{s_I=-\gamma_I(0.55,0,0.45)^T I.}
$$

Ved lading skifter strømmen fortegn og syre produseres.

In [ ]:
c_ref = 4.8  # mol/L, pedagogisk referansekonsentrasjon
gamma_I = 2.2e-5  # mol/(L A s), skalert undervisningsparameter
syre_fordeling = np.array([0.55, 0.0, 0.45])

k_syrespenning = 0.055  # V per relativ konsentrasjonsendring

## D.3 Aktiv overflate

Under utlading blokkeres aktiv overflate gradvis. Under lading kan den delvis gjenopprettes:

$$
\boxed{
\dot a
=-k_a\left(\frac{I_+}{I_{ref}}\right)^{n_P}a
+k_r\frac{I_-}{I_{ref}}(1-a),}
$$

med

$$I_+=\max(I,0),\qquad I_-=\max(-I,0).$$

Den effektive polarisasjonen øker som $1/a$.

In [ ]:
k_a = 1.2e-5   # 1/s
k_r = 3.0e-5   # 1/s
I_ref = 50.0
n_P = 1.25

## D.4 Syreavhengig likevektsspenning

Vi bruker middelverdien i elektrodene:

$$\bar c_e=\frac{c_-+c_+}{2}.$$

Likevektsspenningen blir

$$
\boxed{
U_{eq}(z,c)=U_{eq}(z)+k_c\ln\left(\frac{\bar c_e}{c_{ref}}\right).}
$$

In [ ]:
def U_eq_hybrid(z, c_minus, c_plus):
    c_mean = max(0.5*(c_minus+c_plus), 1e-6)
    return U_eq_soc(z)+k_syrespenning*np.log(c_mean/c_ref)

## Oppgave D1: Implementer hybridmodellen

Bruk strømprofilen

- 80 A utlading i 120 s
- 0 A hvile i 1200 s
- 20 A lading i 300 s
- deretter hvile

RC-polarisasjonen skaleres med $1/a$.

In [ ]:
def strøm_hybrid(t):
    if t < 120.0:
        return 80.0
    if t < 1320.0:
        return 0.0
    if t < 1620.0:
        return -20.0
    return 0.0


def hybrid_ode(t, X):
    z = X[0]
    u = X[1:4]
    c = X[4:7]
    a = X[7]
    I = strøm_hybrid(t)

    dz = ...
    du = A_u@u + b_u*I/max(a, 0.10)

    syrekilde = -gamma_I*syre_fordeling*I
    dc = ...

    I_ut = max(I, 0.0)
    I_lad = max(-I, 0.0)
    da = ...

    return np.concatenate([[dz], du, dc, [da]])

## Oppgave D2: Simuler og beregn terminalspenningen

Start med

$$z=0.80,\quad u=0,\quad c=(c_{ref},c_{ref},c_{ref}),\quad a=1.$$

Plott:

- strøm
- SOC
- tre RC-spenninger
- tre syrekonsentrasjoner
- aktiv overflate
- terminalspenning

In [ ]:
X0_hybrid = np.concatenate([
    [0.80],
    np.zeros(3),
    c_ref*np.ones(3),
    [1.0]
])


def terminalspenning_hybrid(X, I):
    z = X[0]
    u = X[1:4]
    c = X[4:7]
    return U_eq_hybrid(z, c[0], c[2]) - R0*I - np.sum(u)

# Simuler for eksempel 2400 s med et egnet tidssteg.

## Oppgave D3: Sammenlign modellnivåer

Sammenlign terminalspenningen fra:

1. bare $U_{eq}-R_0I$
2. ekvivalent krets
3. ekvivalent krets og SOC
4. full hybridmodell

Hvilken modell beskriver

- det momentane fallet
- sekundresponsen
- lang relaksasjon
- spenningsendring etter syretømming?

# Del E: Egenmoder i hybridmodellens lineære del

Når strømmen er null og $z$ og $a$ holdes faste, er RC- og syredynamikken lineær:

$$
\frac d{dt}
\begin{pmatrix}u\\c-c_{ref}\mathbf1\end{pmatrix}
=
\begin{pmatrix}A_u&0\\0&A_c\end{pmatrix}
\begin{pmatrix}u\\c-c_{ref}\mathbf1\end{pmatrix}.
$$

Systemet har tre elektriske moder, to dempede syreforskjellsmoder og én nullmode for total syre.

## Oppgave E1: Bygg blokkmatrisen

Beregn egenverdier og egenvektorer. Sorter modene etter tidskonstant og tolk dem fysisk.

In [ ]:
A_relaks = np.block([
    [A_u, np.zeros((3, 3))],
    [np.zeros((3, 3)), A_c]
])

relaks_eig, relaks_P = ...
print("Egenverdier:
", relaks_eig)

## Oppgave E2: Direkte og modal relaksasjon

Ta tilstanden ved slutten av utladningspulsen som starttilstand for hvile. Sammenlign:

- direkte Euler på relaksasjonssystemet
- modal løsning
- full ikke-lineær hybridmodell

Forklar hvorfor SOC og total syremengde nesten ikke gjenopprettes under hvile, mens spennings- og konsentrasjonsforskjeller gjør det.

# Fordypning 1: Blyioner og ladningsaksept

Ved lading forbrukes Pb$^{2+}$ av hovedreaksjonen. Oppløsning av PbSO$_4$ forsøker å fylle ionelageret.

For én elektrode bruker vi

$$
\boxed{
\dot c_{Pb}=k_{oppl}(c_{Pb}^*-c_{Pb})-\gamma_{Pb}I_{lad}.}
$$

Konsentrasjonsoverspenningen er

$$
\boxed{
\eta_{Pb}=\frac{RT}{2F}\ln\left(\frac{c_{Pb}^*}{c_{Pb}}\right).}
$$

Når $c_{Pb}$ blir liten, øker ladespenningen raskt og batteriet kan nå spenningsgrensen før SOC er 100 prosent.

In [ ]:
c_Pb_eq = 1.0       # normalisert
k_oppl = 0.015         # 1/s
gamma_Pb = 2.0e-4      # 1/(A s), pedagogisk


def blyion_ode(t, c_Pb, I_lad):
    return k_oppl*(c_Pb_eq-c_Pb)-gamma_Pb*max(-I_lad, 0.0)


def eta_blyion(c_Pb, T=T_ref):
    c_safe = max(c_Pb, 1e-6)
    return R_gass*T/(2*F)*np.log(c_Pb_eq/c_safe)

## Oppgave F1: Spenningsbegrenset lading

Sammenlign ladepulser på 10, 30 og 60 A. Stopp pulsen når celleterminalspenningen når 2.45 V.

Undersøk:

- akseptert ladning
- minimum Pb$^{2+}$-konsentrasjon
- maksimal ionebetinget overspenning
- betydningen av $k_{oppl}$

# Fordypning 2: Gassproduksjon og oksygenrekombinasjon

Ved høy ladespenning kan positiv elektrode produsere oksygen. Oksygenet transporteres gjennom separatoren og kan rekombineres ved negativ elektrode.

Vi bruker et effektivt oksygenlager $n_{O_2}$:

$$
\boxed{
\dot n_{O_2}
=\frac{I_{O_2}}{4F}
-k_{trans}n_{O_2}
-\dot n_{vent}.}
$$

Oksygenutviklingsstrømmen modelleres med et Tafel-lignende uttrykk over en terskelspenning.

In [ ]:
U_gass = 2.38
I0_O2 = 0.08
b_O2 = 0.055
k_O2_transport = 0.025
I_rekom_maks = 8.0


def oksygenstrøm(U):
    if U <= U_gass:
        return 0.0
    return I0_O2*10**((U-U_gass)/b_O2)


def gass_ode(t, n_O2, U):
    I_O2 = oksygenstrøm(U)
    I_rekom = min(4*F*k_O2_transport*n_O2, I_rekom_maks)
    I_vent = max(I_O2-I_rekom-I_rekom_maks, 0.0)
    dn = I_O2/(4*F)-k_O2_transport*n_O2-I_vent/(4*F)
    return dn, I_O2, I_rekom, I_vent

## Oppgave G1: Overladningspuls

Påfør en spenningsprofil som stiger fra 2.30 til 2.50 V. Simuler oksygenlager, rekombinasjonsstrøm og ventilert oksygen.

Diskuter:

- når gassing starter
- hvorfor rekombinasjonen mettes
- hvorfor ventilering gir permanent vannforbruk

## G.2 Vannbalanse

En enkel tapsindikator er

$$
\boxed{
\dot m_{H_2O}
=-k_w(I_{H_2}+I_{O_2,vent}).}
$$

Denne trenger ikke påvirke spenningen i hovedmodellen, men kan brukes som en langsom levetidsindikator.

# Valgfri fordypning: Krystallhukommelse

Nylig dannet PbSO$_4$ kan representeres som lettere oppløselig enn eldre, herdede krystaller:

$$
\dot q_{ny}
=k_fI_{ut}-\frac{q_{ny}}{\tau_h}-k_dI_{lad}q_{ny},
$$

$$
\dot q_{hard}
=\frac{q_{ny}}{\tau_h}-k_{dh}I_{lad}q_{hard}.
$$

Dette gir modellen hukommelse utover SOC. Sammenlign umiddelbar opplading etter utlading med opplading etter lang hvile.

# Modellkritikk

Diskuter minst åtte punkter:

- Ekvivalentkretsen er en matematisk terminalmodell.
- RC-leddene representerer samlede elektrokjemiske prosesser.
- Syremodellen har bare tre romlige volumer.
- Transportkoeffisientene er konstante i hovedmodellen.
- Porøsitet og diffusjon er ikke eksplisitt SOC-avhengige.
- Likevektsspenningen er en pedagogisk funksjon.
- Aktiv overflate er samlet i én tilstand.
- Sulfatkrystallgeometri behandles ikke direkte.
- Blyionmodellen bruker normaliserte konsentrasjoner.
- Butler–Volmer-kinetikk er ikke implementert fullt.
- Gassmodellen bruker en skarp terskel og et effektivt lager.
- Temperatur holdes konstant.
- Vannforbruk påvirker ikke elektrolyttvolumet tilbake.
- Positiv gitterkorrosjon er utelatt.
- Parametrene tilhører ikke et bestemt batteri.
- Euler krever kontroll av tidssteget, særlig for den raske RC-moden.

## Mulige videreføringer

- temperatur som tilstand
- SOC-avhengige motstander og kapasiteter
- porøsitet og Bruggeman-korreksjon
- full Butler–Volmer-kinetikk
- dynamisk gassfase og ventiltrykk
- positiv gitterkorrosjon
- empirisk parameteridentifikasjon
- sammenligning med pulsdata
- romlig diffusjonsmodell i Matematikk 2

# Overgang til Matematikk 2

Trevolumsmodellen er en grov romlig reduksjon. I en mer detaljert modell blir syrekonsentrasjonen et felt:

$$c=c(x,t).$$

En forenklet reaksjons-diffusjonsmodell kan være

$$
\boxed{
\varepsilon\frac{\partial c}{\partial t}
=
\frac{\partial}{\partial x}
\left(D_{eff}(c,z,T)\frac{\partial c}{\partial x}\right)
+s_{reaksjon}.}
$$

Matematikk 2 kan dele elektrodene og separatoren i flere romlige celler og undersøke hvordan trevolumsmodellen fremkommer som den groveste diskretiseringen.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan trevolumsmodellen ga matrisen $B_c^TD_cB_c$,
2. hvorfor syretransportmatrisen hadde en nullmode,
3. hvordan ulike volum ga en ikke-symmetrisk systemmatrise,
4. hvordan RC-leddene beskrev flere spenningsrelaksasjoner,
5. hvordan SOC, syre og aktiv overflate ble koblet,
6. hvorfor terminalspenningen gjenopprettes under hvile uten at SOC gjør det,
7. hvordan blyionmangel kan begrense ladningsaksept,
8. hvordan oksygenproduksjon og rekombinasjon påvirker lading,
9. hvilke egenmoder som var elektriske og hvilke som var kjemiske,
10. hvorfor en mer detaljert syremodell blir en PDE.

## Faglig bakgrunn

Prosjektet er inspirert av en modellfamilie for ventilerte og ventilregulerte blysyrebatterier. Modellfamilien kombinerer impedansbaserte elektriske delmodeller med redusert elektrolytttransport, side­reaksjoner, blyionbegrensning og korttidshistorie. Studentmodellen beholder modellarkitekturen, men bruker færre tilstander og forenklede undervisningsparametre.